# Cross-matching local and HEASARC catalogs using Python

## Learning Goals

By the end of this tutorial, you will:

- Have used the Astroquery Python package.
- Be able to cross-match a local catalog with a HEASARC-hosted catalog.
- Understand how to upload the local catalog so that matching is performed on HEASARC servers.

## Introduction

In this bite-sized tutorial we take you through the process of cross-matching a local
catalog (either stored locally or loaded into local memory) to a catalog hosted by HEASARC.

This demonstration uploads a catalog table to HEASARC's table access protocol (TAP)
service. That then runs an Astronomical Data Query Language (ADQL) query we set up
to find matches between the local and HEASARC catalogs and returns the results.

### Runtime

As of 6th March 2026, this notebook takes ~30 s to run to completion on Fornax using the 'small' server with 8GB RAM/ 2 cores.

## Imports

This notebook uses features from an Astroquery pre-release. You will need to install
the latest version using the command below. We will remove this once Astroquery
v0.4.12 is officially released.

```
pip install --pre astroquery --upgrade
```

In [1]:
import pandas as pd
from astropy.table import Table
from astropy.units import Quantity
from astroquery.heasarc import Heasarc

***

## 1. Prepare our sample for cross-matching to a HEASARC catalog

We assume that you have a catalog of sources available and that you wish to find
matches in one of HEASARC's catalogs. In this instance we also assume that the
catalog is formatted as a 'comma separated values' (CSV) file.

To demonstrate, we use a relatively small set of galaxy clusters from the
SDSSRM-XCS sample ([Giles P. A. et al. 2022](https://ui.adsabs.harvard.edu/abs/2022MNRAS.516.3878G/abstract); [Turner D. J. et al. 2025](https://ui.adsabs.harvard.edu/abs/2025MNRAS.537.1404T/abstract)).

First, we define the path to the CSV file (I know it isn't really a 'local' file, but
you could set this to the path to a CSV on your machine that you wish to cross-match!):

In [2]:
# URL of a sample file
samp_path = (
    "https://github.com/DavidT3/XCS-Mass-II-Analysis/raw/refs/heads/main/"
    "sample_files/SDSSRM-XCS_base_sample.csv"
)

You might have noticed that we imported the 'Pandas' module in the
['Imports Section'](#imports) - we're going to use it to read our CSV file into
a Pandas DataFrame.

As we've pointed out, the path to the file we're using in this example is actually a
URL, but the `read_csv()` function can handle both remote and local files.

In [3]:
# Load the CSV file using Pandas
samp = pd.read_csv(samp_path)
samp

,name,MEM_MATCH_ID,xapa_ra,xapa_dec,rm_ra,rm_dec,z,r500,r500-,r500+,richness,richness_err,XCS_NAME,R_LAMBDA,xmm_targeted,xmm_serendipitous
0,SDSSXCS-124,124,0.80058,-6.09182,0.798261,-6.091694,0.2475,1181.028,21.202,23.203,109.550,4.490,XMMXCS J000312.1-060530.5,1.018410,True,False
1,SDSSXCS-2789,2789,0.95554,2.06802,0.956981,2.066469,0.1053,1007.861,17.194,17.202,38.904,2.830,XMMXCS J000349.3+020404.8,0.827942,True,False
2,SDSSXCS-290,290,2.72264,29.16102,2.714137,29.161154,0.3485,913.052,30.879,31.210,105.096,5.994,XMMXCS J001053.4+290939.6,1.009990,True,False
3,SDSSXCS-1018,1018,4.40633,-0.87619,4.406711,-0.878340,0.2144,902.259,22.445,23.366,56.997,3.219,XMMXCS J001737.5-005234.2,0.893655,False,True
4,SDSSXCS-134,134,4.90839,3.60982,4.911069,3.599257,0.2773,1123.321,19.219,19.226,108.604,4.792,XMMXCS J001938.0+033635.3,1.016645,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,SDSSXCS-2092,2092,350.35308,19.75422,350.357721,19.752055,0.3282,781.325,32.933,33.421,57.317,3.998,XMMXCS J232124.7+194515.1,0.894658,False,True
146,SDSSXCS-17923,17923,350.47041,19.89360,350.487259,19.848230,0.3149,488.755,61.669,85.825,29.431,3.110,XMMXCS J232152.8+195336.9,0.782999,False,True
147,SDSSXCS-68,68,354.41088,0.27071,354.415537,0.271367,0.3006,1181.483,29.207,29.247,126.331,4.887,XMMXCS J233738.6+001614.5,1.047856,True,False
148,SDSSXCS-147,147,355.31937,-9.02468,355.320924,-9.019937,0.2594,1143.058,23.681,25.081,104.470,4.513,XMMXCS J234116.6-090128.8,1.008785,True,False


```{caution}
The query we submit in [Section 4](#4-run-the-cross-match-and-retrieve-the-results) can
be sensitive to certain symbols in column names. Having a column name with a '-'
symbol, for instance, will cause an error.

We also note that queries are **not** case-sensitive, so having columns
named 'e_kT' and 'E_kT' to indicate non-symmetrical uncertainties, for instance, would
trigger an error message about duplicate column names.
```

As per our warning above, some of the column names in our sample would cause errors when
we try to upload them to the HEASARC TAP service, so we'll replace the offending
symbols with something else:

In [4]:
mod_samp_cols = samp.columns.str.replace("-", "minus")
mod_samp_cols = mod_samp_cols.str.replace("+", "plus")

samp.columns = mod_samp_cols

Now we will convert our Pandas DataFrame to an Astropy Table, which we'll be able to
pass directly to an Astroquery query function in [Section 4](#4-run-the-cross-match-and-retrieve-the-results):

In [5]:
# Use a useful astropy table method to convert to Pandas dataframe
samp_tab = Table.from_pandas(samp)
samp_tab

name,MEM_MATCH_ID,xapa_ra,xapa_dec,rm_ra,rm_dec,z,r500,r500minus,r500plus,richness,richness_err,XCS_NAME,R_LAMBDA,xmm_targeted,xmm_serendipitous
str13,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,str25,float64,bool,bool
SDSSXCS-124,124,0.80058,-6.09182,0.7982613503413916,-6.091694414143023,0.2475,1181.028,21.202,23.203,109.55,4.49,XMMXCS J000312.1-060530.5,1.01841,True,False
SDSSXCS-2789,2789,0.95554,2.06802,0.9569814053031678,2.0664686270472927,0.1053,1007.861,17.194,17.202,38.904,2.83,XMMXCS J000349.3+020404.8,0.82794166,True,False
SDSSXCS-290,290,2.72264,29.16102,2.714136576083092,29.16115396854516,0.3485,913.052,30.879,31.21,105.096,5.994,XMMXCS J001053.4+290939.6,1.0099899,True,False
SDSSXCS-1018,1018,4.40633,-0.87619,4.406711392790498,-0.8783404073160839,0.2144,902.259,22.445,23.366,56.997,3.219,XMMXCS J001737.5-005234.2,0.8936554,False,True
SDSSXCS-134,134,4.90839,3.60982,4.911069186700388,3.5992569126673364,0.2773,1123.321,19.219,19.226,108.604,4.792,XMMXCS J001938.0+033635.3,1.0166453000000002,True,False
SDSSXCS-119,119,8.4647,-7.8629,8.471426198257149,-7.869543305681293,0.3042,944.275,46.422,46.747,128.192,5.697,XMMXCS J003351.5-075146.4,1.0509257,True,False
SDSSXCS-209,209,9.27682,9.15719,9.27850558294756,9.156703629803651,0.2688,1295.732,27.424,27.434,101.709,4.746,XMMXCS J003706.4+090925.8,1.0033951,True,False
SDSSXCS-15,15,11.62825,20.46769,11.622199893045266,20.46801265525455,0.104,850.081,63.426,73.274,123.356,3.702,XMMXCS J004630.7+202803.6,1.0428745,True,False
SDSSXCS-71,71,13.99656,26.3303,14.001076223842912,26.34229673287753,0.1961,1095.927,19.708,19.728,112.922,4.166,XMMXCS J005559.1+261949.0,1.0246024,True,False


You might have a catalog stored as a FITS, ASCII, or even HDF5 file - as long as
it is in the form of an Astropy Table by this point, the rest of the steps in this
demonstration should work.

## 2. The HEASARC TAP service

HEASARC, along with many other virtual observatory (VO) services, offers a table
access protocol (TAP) service. That means that we can perform operations on
HEASARC-hosted tables by constructing ADQL queries.

The HEASARC TAP service supports the **upload** of tables for use by queries, which we
will be using for this demonstration; note, however, that not all VO services support
table upload.

We will use the [Astroquery](https://github.com/astropy/astroquery) Python package to
interact with the HEASARC TAP service in this tutorial. Specifically, the
`Heasarc.query_tap(...)` method:

In [6]:
# Show the docstring for the query_tap method
help(Heasarc.query_tap)

Help on method query_tap in module astroquery.heasarc.core:

query_tap(query, *, maxrec=None, uploads=None) method of astroquery.heasarc.core.HeasarcClass instance
    Send query to HEASARC's Xamin TAP using ADQL.
    Results in `~pyvo.dal.TAPResults` format.
    result.to_table gives `~astropy.table.Table` format.

    Parameters
    ----------
    query : str
        ADQL query to be executed
    maxrec : int
        maximum number of records to return
    uploads : dict
        a mapping from table names used in the query to file like
        objects containing a votable
        (e.g. a file path or `~astropy.table.Table`).

    Returns
    -------
    result : `~pyvo.dal.TAPResults`
        TAP query result.
    result.to_table : `~astropy.table.Table`
        TAP query result as `~astropy.table.Table`
    result.to_qtable : `~astropy.table.QTable`
        TAP query result as `~astropy.table.QTable`


## 3. Construct a matching query

For this demonstration, we're assuming that you already have a HEASARC-hosted catalog
in mind; if not, you might find the
'{doc}`Find specific HEASARC catalogs using Python <finding_relevant_heasarc_catalog>`'
tutorial useful.

We are going to cross-match our sample to the 'Second ROSAT all-sky survey' source
catalog (2RXS; [Boller T. et al. 2016](https://ui.adsabs.harvard.edu/abs/2016A%26A...588A.103B/abstract)). HEASARC's table name for this catalog is:

In [7]:
heasarc_cat_name = "rass2rxs"

Now we must decide how close a 2RXS entry has to be to a source in our sample to be
considered a match. As we're demonstrating using a sample of galaxy
clusters (extended objects), we choose a fairly large matching distance - you
should adjust this based on your own use case:

In [8]:
match_dist = Quantity(2, "arcmin")

There are two sets of RA-Dec coordinates for each entry in our local catalog. The
difference between them is irrelevant to this demonstration, but it is an important
reminder that you will likely have to adjust the column names for your local
catalog in the ADQL query.

To make that a little easier, we define the RA/Dec column names we want to use here:

In [9]:
local_ra_col = "rm_ra"
local_dec_col = "rm_dec"

Now we construct a simple ADQL query that will return all columns (`SELECT *`) and
entries where (`WHERE` - unsurprisingly) the coordinate of a 2RXS
entry (`point('ICRS',cat.ra,cat.dec)`) is within (`contains(...)`) a circle with
radius `match_dist` centered on a source in our
sample (`circle('ICRS',loc.{lra},loc.{ldec},{md})`):

In [10]:
query = (
    "SELECT * "
    "FROM {hcn} as cat, tap_upload.local_samp as loc "
    "WHERE "
    "contains(point('ICRS',cat.ra,cat.dec), "
    "circle('ICRS',loc.{lra},loc.{ldec},{md}))=1".format(
        md=match_dist.to("deg").value.round(4),
        lra=local_ra_col,
        ldec=local_dec_col,
        hcn=heasarc_cat_name,
    )
)

query

"SELECT * FROM rass2rxs as cat, tap_upload.local_samp as loc WHERE contains(point('ICRS',cat.ra,cat.dec), circle('ICRS',loc.rm_ra,loc.rm_dec,0.0333))=1"

```{seealso}
A general tutorial on the many uses and features of ADQL is out of the scope of this
bite-sized demonstration. Various resource for learning ADQL are available online, such
as [this short course](https://docs.g-vo.org/adql/) ([Demleitner M. and Heinl H. 2024](https://dc.g-vo.org/voidoi/q/lp/custom/10.21938/uH0_xl5a6F7tKkXBSPnZxg)),
or the NASA Astronomical Virtual Observatories (NAVO)
[catalog queries tutorial](https://nasa-navo.github.io/navo-workshop/content/reference_notebooks/catalog_queries.html).
```

## 4. Run the cross-match and retrieve the results

Finally, we can run our cross-match!

The query we designed will be passed to the HEASARC TAP service through the
Astroquery module.

When we run this query, we also upload our sample table (prepared in
[Section 1](#1-prepare-our-sample-for-cross-matching-to-a-heasarc-catalog)). Notice
that the key we assign our table in the `uploads` dictionary is the same as the table
name in the ADQL query we prepared in [Section 3](#3-construct-a-matching-query).

In [11]:
cat_match = Heasarc.query_tap(query, uploads={"local_samp": samp_tab})

Note that the `Heasarc.query_tap(...)` call submits a **synchronous** query to the
HEASARC TAP service, as opposed to an ***asynchronous*** query.

[A discussion of the differences can be found here](https://pyvo.readthedocs.io/en/stable/dal/#synchronous-vs-asynchronous-query), but
the summary is that a synchronous query will stay connected to the HEASARC service until the table
operation is complete and the results are returned, whereas submitting a query asynchronously will send
the job to HEASARC, get a URL reporting the status of the job in return, and then that URL
will have to be polled to find out when the results are ready.

Asynchronous submission is preferable for long-running queries, as it won't be
sensitive to any network issues that might occur while waiting for the results like
a synchronous query would be. Asynchronous HEASARC queries cannot currently be
submitted through Astroquery, though you could instead use the [PyVO module](https://github.com/astropy/pyvo).

Our match results have been returned, and we can convert them into an Astropy Table object, as they
can be a little easier to work with than the `TAPResults` object we received.

By putting the variable name at the bottom of the code cell, we can see a nice rendered
version of the table (only in a Jupyter notebook environment) and see that we do
have some matches!

In [12]:
cat_match_tab = cat_match.to_table()
# Visualize the table
cat_match_tab

cat___row,cat_entry_number,cat_name,cat_skyfield_number,cat_skyfield_source_number,cat_detection_likelihood,cat_counts,cat_counts_error,cat_count_rate,cat_count_rate_error,cat_exposure,cat_ra,cat_dec,cat_lii,cat_bii,cat_lambda,cat_beta,cat_source_extent,cat_source_extent_error,cat_source_extent_prob,cat_hardness_ratio_1,cat_hardness_ratio_1_error,cat_hardness_ratio_2,cat_hardness_ratio_2_error,cat_unique_flag,cat_extended_region_flag,cat_nearby_src_det_flag,cat_source_quality_flag,cat_max_amplitude,cat_mean_count_rate,cat_mean_count_rate_error,cat_lc_counts,cat_min_count_rate,cat_min_count_rate_error,cat_max_count_rate,cat_max_count_rate_error,cat_lc_chi2,cat_excess_variance,cat_excess_variance_error,cat_excess_variance_sigma,cat_nh_gal,cat_plaw_nh,cat_plaw_nh_error,cat_plaw_norm,cat_plaw_norm_error,cat_plaw_photon_index,cat_plaw_photon_index_error,cat_plaw_count_rate,cat_plaw_flux,cat_plaw_chi2_reduced,cat_plaw_chi2,cat_plaw_number_data_pts,cat_plaw_dof,cat_mekal_nh,cat_mekal_nh_error,cat_mekal_norm,cat_mekal_norm_error,cat_mekal_temperature,cat_mekal_temperature_error,cat_mekal_count_rate,cat_mekal_flux,cat_mekal_chi2_reduced,cat_mekal_chi2,cat_mekal_number_data_pts,cat_mekal_dof,cat_bb_nh,cat_bb_nh_error,cat_bb_norm,cat_bb_norm_error,cat_bb_temperature,cat_bb_temperature_error,cat_bb_count_rate,cat_bb_flux,cat_bb_chi2_reduced,cat_bb_chi2,cat_bb_number_data_pts,cat_bb_dof,cat_x_pixel,cat_x_pixel_error,cat_y_pixel,cat_y_pixel_error,cat_x_sky_pixel,cat_y_sky_pixel,cat_extraction_radius,cat_extraction_radius_frac,cat_total_photons,cat_bkg_in_extr_reg,cat_vignetting_factor,cat_remarks,cat_band_a_tot_counts,cat_band_b_tot_counts,cat_band_c_tot_counts,cat_band_d_tot_counts,cat_band_a_bkg_counts,cat_band_b_bkg_counts,cat_band_c_bkg_counts,cat_band_d_bkg_counts,cat_remaining_bkg_area,cat_band_a_counts,cat_band_b_counts,cat_band_c_counts,cat_band_d_counts,cat_xmmsl1_number_ctrprts,cat_xmmsl1_nearest,cat_xmmsl1_separation,cat_xmmsl1_name,cat_xmmsl1_ra,cat_xmmsl1_dec,cat_xmmsl1_bb_count_rate,cat_xmmsl1_bb_count_rate_err,cat_xmmsl1_sb_count_rate,cat_xmmsl1_sb_count_rate_err,cat_threexmm_number_ctrprts,cat_threexmm_nearest,cat_threexmm_separation,cat_threexmm_name,cat_threexmm_ra,cat_threexmm_dec,cat_threexmm_count_rate,cat_threexmm_count_rate_err,cat_threexmm_flux,cat_threexmm_flux_error,cat_tworxp_number_ctrprts,cat_tworxp_nearest,cat_tworxp_separation,cat_tworxp_name,cat_tworxp_ra,cat_tworxp_dec,cat_tworxp_count_rate,cat_tworxp_count_rate_error,cat_tworxp_exposure,cat_tworxp_obsid,cat_onerxs_number_ctrprts,cat_onerxs_nearest,cat_onerxs_separation,cat_onerxs_name,cat_onerxs_ra,cat_onerxs_dec,cat_onerxs_count_rate,cat_onerxs_count_rate_error,cat_onerxs_counts,cat_onerxs_counts_error,cat_onerxs_det_likelihood,cat_onerxs_exposure,cat_onerxs_hr_1,cat_onerxs_hr_1_error,cat_onerxs_hr_2,cat_onerxs_hr_2_error,cat_veron_number_ctrprts,cat_veron_nearest,cat_veron_separation,cat_veron_name,cat_veron_type,cat_veron_vmag,cat_veron_redshift,cat_veron_source_number,cat_veron_ra,cat_veron_dec,cat_tycho2_number_ctrprts,cat_tycho2_nearest,cat_tycho2_separation,cat_tycho2_ra,cat_tycho2_dec,cat_tycho2_source_number,cat_tycho2_vmag,cat_tycho2_bmag,cat_tycho2_tyc1_number,cat_tycho2_tyc2_number,cat_tycho2_tyc3_number,cat_bsc_number_ctrprts,cat_bsc_nearest,cat_bsc_separation,cat_bsc_ra,cat_bsc_dec,cat_bsc_vmag,cat_bsc_spect_type,cat_bsc_source_number,cat_hd_source_number,cat_hmxb_number_ctrprts,cat_hmxb_nearest,cat_hmxb_separation,cat_hmxb_name,cat_hmxb_alt_name,cat_hmxb_ra,cat_hmxb_dec,cat_hmxb_vmag,cat_lmxb_number_ctrprts,cat_lmxb_nearest,cat_lmxb_separation,cat_lmxb_name,cat_lmxb_alt_name,cat_lmxb_ra,cat_lmxb_dec,cat_lmxb_vmag,cat_atnf_number_ctrprts,cat_atnf_nearest,cat_atnf_separation,cat_atnf_name,cat_atnf_ra,cat_atnf_dec,cat_atnf_pulsar_type,cat_atnf_pulse_period,cat_fuhr_number_ctrprts,cat_fuhr_nearest,cat_fuhr_separation,cat_fuhr_name,cat_fuhr_ra,cat_fuhr_dec,cat_fuhr_source_number,cat_onesxps_number_ctrprts,cat_onesxps_nearest,cat_onesxps_separation

Seeing as there are a lot of columns in the results table (all the columns from the
uploaded table and the 2RXS catalog), we can check what columns are available by
accessing the `colnames` property of our table:

In [13]:
cat_match_tab.colnames

['cat___row',
 'cat_entry_number',
 'cat_name',
 'cat_skyfield_number',
 'cat_skyfield_source_number',
 'cat_detection_likelihood',
 'cat_counts',
 'cat_counts_error',
 'cat_count_rate',
 'cat_count_rate_error',
 'cat_exposure',
 'cat_ra',
 'cat_dec',
 'cat_lii',
 'cat_bii',
 'cat_lambda',
 'cat_beta',
 'cat_source_extent',
 'cat_source_extent_error',
 'cat_source_extent_prob',
 'cat_hardness_ratio_1',
 'cat_hardness_ratio_1_error',
 'cat_hardness_ratio_2',
 'cat_hardness_ratio_2_error',
 'cat_unique_flag',
 'cat_extended_region_flag',
 'cat_nearby_src_det_flag',
 'cat_source_quality_flag',
 'cat_max_amplitude',
 'cat_mean_count_rate',
 'cat_mean_count_rate_error',
 'cat_lc_counts',
 'cat_min_count_rate',
 'cat_min_count_rate_error',
 'cat_max_count_rate',
 'cat_max_count_rate_error',
 'cat_lc_chi2',
 'cat_excess_variance',
 'cat_excess_variance_error',
 'cat_excess_variance_sigma',
 'cat_nh_gal',
 'cat_plaw_nh',
 'cat_plaw_nh_error',
 'cat_plaw_norm',
 'cat_plaw_norm_error',
 'cat_pla

Then we could pull out the values of a particular column as a Numpy array:

In [14]:
cat_match_tab["cat_count_rate"].value.data

array([0.1269, 0.1571, 0.2034, 0.0537, 0.0344, 0.2065, 0.047 , 0.1215,
       0.0404, 0.1923, 0.2141, 0.0808, 0.0779, 0.2672, 0.112 , 0.0226,
       0.0499, 0.0664, 0.076 , 0.2462, 0.1051, 0.0989, 0.046 , 0.083 ,
       0.2597, 0.1892, 0.2743, 0.3465, 0.0816, 0.0907, 0.0351, 0.3021,
       0.1367, 0.4966, 0.024 , 0.1641, 0.471 , 0.6951, 0.0399, 0.0281,
       0.0367, 0.0521, 0.0654, 0.104 , 0.0664, 0.0416, 0.1981, 0.0606,
       0.8752, 0.1035, 0.0415, 0.1403, 0.0696, 0.0716, 0.1358, 0.2921,
       0.6902, 0.0228, 0.0394, 0.9201, 0.0229, 0.0667, 0.3266, 0.2542,
       0.433 , 0.6581, 0.1356, 0.1161, 0.0478, 0.0876, 0.2585, 0.0292,
       0.1205, 0.0892, 0.0194, 0.551 , 0.524 , 0.351 , 0.3377, 0.0668,
       0.1342, 0.0952, 0.315 , 0.1736, 0.0883, 0.0466, 0.1052, 0.0269,
       0.0509, 0.0211, 0.183 , 0.1066, 0.0223])

```{seealso}
An additional resource for learning about the use of virtual observatory services
is the [NASA Astronomical Virtual Observatories (NAVO) workshop notebook set](https://nasa-navo.github.io/navo-workshop/).

[Section 3 of the 'Catalog Queries' notebook](https://nasa-navo.github.io/navo-workshop/content/reference_notebooks/catalog_queries.html#using-the-tap-to-cross-correlate-and-combine) is particularly relevant to this bite-sized tutorial.
```

## About this notebook

Author: David Turner, HEASARC Staff Scientist

Updated On: 2026-03-06

### Additional Resources

Support: [HEASARC Helpdesk](https://heasarc.gsfc.nasa.gov/cgi-bin/Feedback?selected=heasarc)

[Short Course on ADQL Website](https://docs.g-vo.org/adql/)

[NAVO Workshop](https://nasa-navo.github.io/navo-workshop/)

[NAVO catalog queries tutorial](https://nasa-navo.github.io/navo-workshop/content/reference_notebooks/catalog_queries.html#using-the-tap-to-cross-correlate-and-combine)

[Astroquery GitHub Repository](https://github.com/astropy/astroquery)

[Latest Astroquery Documentation](https://astroquery.readthedocs.io/en/latest/)

[Description of synchronous and asynchronous queries](https://pyvo.readthedocs.io/en/stable/dal/#synchronous-vs-asynchronous-query)

### Acknowledgements


### References

[Giles P. A., Romer A. K., Wilkinson R., Bermeo A., Turner D. J. et al. (2022)](https://ui.adsabs.harvard.edu/abs/2022MNRAS.516.3878G/abstract) - _The XMM Cluster Survey analysis of the SDSS DR8 redMaPPer catalogue: implications for scatter, selection bias, and isotropy in cluster scaling relations_

[Turner D. J., Giles P. A., Romer A. K., Pilling J., Lingard T. K. et al. (2025)](https://ui.adsabs.harvard.edu/abs/2025MNRAS.537.1404T/abstract) - _The XMM Cluster Survey: automating the estimation of hydrostatic mass for large samples of galaxy clusters ─ I. Methodology, validation, and application to the SDSSRM-XCS sample_

[Boller T., Freyberg M.J., Trümper J. et al. (2016)](https://ui.adsabs.harvard.edu/abs/2016A%26A...588A.103B/abstract) - _Second ROSAT all-sky survey (2RXS) source catalogue_